## XGBoost Regressor Model

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta) 
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df.isnull().sum()

om              0
yr              0
mo              0
dy              0
date            0
time            0
tz              0
datetime_utc    0
st              0
stf             0
mag             0
inj             0
fat             0
loss            0
slat            0
slon            0
elat            0
elon            0
len             0
wid             0
ns              0
sn              0
f1              0
f2              0
f3              0
f4              0
fc              0
dtype: int64

In [3]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor  # Este podrías quitarlo si ya no lo usas

# Modelo de XGBoost Regressor
# Definir X y y (asegúrate de que ya tienes estas variables previamente definidas)
X = df[['mag', 'slat', 'slon', 'elat', 'elon', 'len', 'wid', 'f1', 'f2', 'f3', 'f4', 'loss']]
y = df['inj']

# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBRegressor().fit(X_train, y_train)
print("Training set score: {:.2f}".format(xgb.score(X_train, y_train)))
print("Test set score: {:.2f}".format(xgb.score(X_test, y_test)))

# Ver la importancia de las variables
coeficientes = pd.Series(xgb.feature_importances_, index=X_train.columns)

# Mostrar la importancia de las características
print("Importancia de las características del modelo XGBoost:")
print(coeficientes)


Training set score: 0.97
Test set score: 0.17
Importancia de las características del modelo XGBoost:
mag     0.213219
slat    0.049334
slon    0.155595
elat    0.044708
elon    0.051663
len     0.037071
wid     0.065230
f1      0.078511
f2      0.068045
f3      0.025773
f4      0.044502
loss    0.166348
dtype: float32


## Metricas para XGBoost Regressor

In [4]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump, load
from time import time
from xgboost import XGBRegressor  # Importar XGBRegressor
from sklearn.model_selection import GridSearchCV

# ------------------------
# Paso 2: Verificar si el modelo ya está guardado
# ------------------------
# Asegurarse de que el modelo esté cargado
try:
    grid_xgb = load('grid_xgb.joblib')  # Intentar cargar el modelo entrenado de XGBoost
    print("Modelo XGBoost cargado correctamente.")
except FileNotFoundError:
    print("No se encontró el modelo guardado. Entrenando el modelo desde cero...")
    
    # Si no se encuentra el modelo, entrenarlo desde cero
    # Aquí debes definir X_train, y_train antes de este paso

    xgb_model = XGBRegressor()
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 6, 10]}  # Parametros para búsqueda
    grid_xgb = GridSearchCV(xgb_model, param_grid, cv=5)
    
    # Entrenar el modelo
    start_time = time()
    grid_xgb.fit(X_train, y_train)
    training_time_xgb = time() - start_time
    
    # Guardar el modelo entrenado
    dump(grid_xgb, 'grid_xgb.joblib')
    print("Modelo entrenado y guardado como 'grid_xgb.joblib'.")

# ------------------------
# Paso 3: Medir el tiempo de entrenamiento
# ------------------------
if 'training_time_xgb' not in globals():
    start_time = time()
    grid_xgb.fit(X_train, y_train)
    training_time_xgb = time() - start_time
else:
    training_time_xgb = 0

# ------------------------
# Paso 4: Hacer predicciones con el modelo XGBoost
# ------------------------
# Predicciones con el mejor modelo de XGBoost
y_pred_xgb = grid_xgb.best_estimator_.predict(X_test)

# ------------------------
# Paso 5: Calcular las métricas para XGBoost
# ------------------------
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

# Ljung-Box p-value para XGBoost
lb_test_xgb = acorr_ljungbox(y_test - y_pred_xgb, lags=[10])
lb_p_value_xgb = lb_test_xgb['lb_pvalue'].iloc[0]

# Jarque-Bera p-value para XGBoost
jb_p_value_xgb = jarque_bera(y_test - y_pred_xgb)[1]

# ------------------------
# Paso 6: Crear DataFrames con los resultados para XGBoost
# ------------------------
# Resultados para XGBoost
resultados_xgb = pd.DataFrame({
    'Modelo': ['XGBoost'],
    'MAPE': [f"{mape_xgb:.2f}"],
    'RMSE': [f"{rmse_xgb:.2f}"],
    'R Cuadrado': [f"{r2_xgb:.2f}"],
    'Ljung-Box Test p-value': [f"{lb_p_value_xgb:.4f}"],
    'Jarque-Bera p-value': [f"{jb_p_value_xgb:.4f}"],
    'CPU time (s)': [round(training_time_xgb, 2)]
})

# ------------------------
# Paso 7: Mostrar los resultados
# ------------------------
print("Métricas XGBoost:")
display(resultados_xgb)


No se encontró el modelo guardado. Entrenando el modelo desde cero...
Modelo entrenado y guardado como 'grid_xgb.joblib'.
Métricas XGBoost:


,Modelo,MAPE,RMSE,R Cuadrado,Ljung-Box Test p-value,Jarque-Bera p-value,CPU time (s)
0,XGBoost,2301408565002240.00,19.63,0.27,0.9042,0.0000,0
